# Part 3. 배포 준비 — 프로젝트 구조 정리

로컬에서 완성한 Streamlit 앱을 GitHub에 올리기 위한 파일들을 준비한다.

```
my-rag-app/          ← 저장소 루트
├── app.py           ← Streamlit 진입 파일 (UI와 호출)
├── requirements.txt ← 배포 시 설치할 패키지 목록
├── .streamlit/
│   └── secrets.toml ← 로컬 키 파일 (업로드 X)
├── .gitignore       ← 올리면 안 되는 것 목록
├── data/            ← 로컬 PDF·캐시 (필요 시)
└── .env             ← 로컬 환경변수 (업로드 X)
```

> `.env`와 `secrets.toml`은 **절대 GitHub에 올라가면 안 된다.**  
> API 키가 노출되면 며칠 안에 봇이 긁어가 과금 폭탄 → Google이 키를 무효화할 수 있다.

## Step 1. .gitignore 생성

`.gitignore`는 Git이 추적하지 않을 파일·폴더를 지정한다.

| 항목 | 정체 | 빼야 하는 이유 |
|------|------|----------------|
| `.env` | 로컬 환경변수 | API 키 노출 → 과금/도용 |
| `.streamlit/secrets.toml` | 로컬 시크릿 | 위와 동일 — 절대 공개 금지 |
| `__pycache__/` | Python 바이트코드 캐시 | 재생성됨 — 저장소만 더러워짐 |
| `.venv/` · `venv/` | 가상환경 폴더 | OS·경로 의존 — 협업자 환경에서 깨짐 |
| `*.pyc` | 컴파일된 파일 | 용량만 차지, 추적 불필요 |

In [ ]:
# .gitignore 파일 생성
gitignore_content = """# 환경변수 — API 키 포함, 절대 업로드 금지
.env

# Streamlit 시크릿 — 마찬가지로 업로드 금지
.streamlit/secrets.toml

# Python 캐시
__pycache__/
*.pyc

# 가상환경
.venv/
venv/

# 로컬 PDF 캐시 (필요 시 주석 해제)
# data/
"""

with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print(".gitignore 생성 완료")
print(gitignore_content)

## Step 2. requirements.txt 생성

배포 환경에서 `pip install -r requirements.txt` 한 줄로 모든 패키지를 설치할 수 있게 한다.

| 명령어 | 특징 |
|--------|------|
| `pip freeze > requirements.txt` | 다운로드 주소(`@`)가 포함되기도 함 |
| `pip list --format=freeze > requirements.txt` | 패키지 이름·버전만 깔끔하게 기록 |

In [ ]:
import subprocess, sys

# pip list --format=freeze 방식으로 requirements.txt 생성
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze"],
    capture_output=True, text=True
)

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(result.stdout)

# 앞 10줄 미리보기
lines = result.stdout.strip().split("\n")
print(f"총 {len(lines)}개 패키지")
print("\n".join(lines[:10]), "\n...")

## Step 3. .streamlit/secrets.toml 생성

In [ ]:
import os

# .streamlit 폴더가 없으면 생성
os.makedirs(".streamlit", exist_ok=True)

# secrets.toml 템플릿 생성 (실제 키는 직접 입력)
secrets_content = """# .streamlit/secrets.toml
# 이 파일은 .gitignore에 등록되어 있어 GitHub에 올라가지 않는다.

# GOOGLE_API_KEY = "여기에_실제_API_키_입력"
"""

secrets_path = ".streamlit/secrets.toml"
with open(secrets_path, "w", encoding="utf-8") as f:
    f.write(secrets_content)

print(f"{secrets_path} 생성 완료")
print(secrets_content)

## Step 4. st.secrets로 불러오기

```python
import streamlit as st

# 평탄(flat) — 키 이름 하나로 바로 접근
api_key = st.secrets["GOOGLE_API_KEY"]
```

## Step 5. .env(로컬) vs st.secrets(배포) 전환

In [ ]:
import os
from dotenv import load_dotenv
import streamlit as st

# 로컬: .env에서 읽기
load_dotenv()
api_key_local = os.getenv("GOOGLE_API_KEY")
print("로컬(.env):", "설정됨" if api_key_local else "없음")

print()
api_key = st.secrets["GOOGLE_API_KEY"]

# 로컬·배포 환경 모두 대응하는 fallback 패턴
print()
# fallback_code = '''
def get_api_key():
    # 배포(Streamlit Cloud): st.secrets 우선
    try:
        return st.secrets["GOOGLE_API_KEY"] # 평탄 방식
    except (KeyError, FileNotFoundError):
        pass
    # 로컬: .env에서 읽기
    return os.getenv("GOOGLE_API_KEY")
# '''
# print(fallback_code)

## Step 6. GitHub에 올리기

> 이미 Git을 쓰고 있었다면 **4~6단계**만 실행하면 된다.

```bash
# 1. 저장소 초기화 (처음 한 번만)
git init

# 2. 변경 사항 스테이징
git add .

# 3. 첫 커밋
git commit -m "deploy ready"

# 4. GitHub에서 빈 repo 생성
#    → github.com에서 "New repository" 클릭

# 5. 원격 저장소 연결
git remote add origin https://github.com/<id>/<repo>.git

# 6. 메인 브랜치로 푸시
git push -u origin main
```

In [ ]:
# 현재 git 상태 및 .gitignore 적용 여부 확인
import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = r.stdout.strip() or r.stderr.strip()
    print(f"$ {cmd}")
    print(output if output else "(출력 없음)")
    print()

run("git status")

# .gitignore가 실제로 파일을 제외하는지 확인
files_to_check = [".env", ".streamlit/secrets.toml", "__pycache__", "requirements.txt"]
print(".gitignore 적용 확인")
print("-" * 40)
for f in files_to_check:
    r = subprocess.run(f'git check-ignore -q "{f}"', shell=True, capture_output=True)
    status = "제외됨 (안전)" if r.returncode == 0 else "추적됨"
    print(f"{f:35s} {status}")

## 정리

| 파일 | 역할 | GitHub 업로드 |
|------|------|:--------------:|
| `app.py` | Streamlit 진입 파일 | O |
| `requirements.txt` | 패키지 목록 | O |
| `.gitignore` | 제외 목록 | O |
| `.streamlit/secrets.toml` | 로컬 API 키 | X |
| `.env` | 로컬 환경변수 | X |
| `data/` | 로컬 PDF | 선택 |

### Streamlit Cloud 배포 시 API 키 입력 경로
1. [share.streamlit.io](https://share.streamlit.io) 에서 앱 배포
2. **Settings → Secrets** 에 `secrets.toml` 내용 붙여넣기
3. 코드에서 `st.secrets["API"]["API_KEY"]` 또는 `st.secrets["GOOGLE_API_KEY"]`로 읽기